# Lecture 16 — Install LIBERO-Plus in a separate runtime, try VLANeXt with "rewritten / clean / empty command"

**First:** open a NEW notebook runtime (do NOT reuse Lecture 12's runtime: there the original `libero` package is installed with `pip install ./third_party/LIBERO`; LIBERO-Plus uses the same name, `libero`). Runtime type **T4 GPU**. Kaggle: Accelerator **GPU T4 x2**, internet on; change the paths `/content` → `/kaggle/working`.

Flow: (1) hardware + disk → (2) system libraries → (3) VLANeXt + LIBERO-Plus (WITHOUT installing it with pip, via `PYTHONPATH`) → (4) `~/.libero_plus/config.yaml` → (5) patches (Lecture 12 + Lecture 16) → (6) assets (6.4 GB zip) → (7) checkpoint (16.4 GB) → (8) smoke test → (9) task selection → (10) conditions: `rewritten` (LIBERO-Plus's own command) and `empty` (empty command); optionally `clean` and `tr` → (11) summary + video.

Disk budget: ~16.4 (checkpoint) + 6.4 (zip; deleted after unpacking) + unpacked assets (size unknown — cell 6 measures it) + ~4.5 (Qwen3-VL-2B) GB. Times are hypotheses; write the measured values in the chat.

In [ ]:
# 1) Hardware + disk: cc (7, 5) = T4; at least ~40 GB of free disk is needed
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, psutil, shutil
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0), "| cc", torch.cuda.get_device_capability(0))
print("RAM %.1f GB | disk free %.0f GB" % (psutil.virtual_memory().total/1e9, shutil.disk_usage("/").free/1e9))

In [ ]:
# 2) System libraries: MuJoCo EGL (COMMON_ISSUES #1) + ImageMagick (wand) for LIBERO-Plus's image perturbations
!apt-get -qq update && apt-get -qq install -y libgl1-mesa-dev libegl1-mesa-dev libxrandr2 libxcursor1 libxinerama1 libxrender1 \
    libexpat1 libfontconfig1-dev libmagickwand-dev > /dev/null && echo "apt ok"
import os; os.environ["MUJOCO_GL"] = "egl"; os.environ["PYOPENGL_PLATFORM"] = "egl"

In [ ]:
# 3) VLANeXt + LIBERO-Plus. We do not pip-install LIBERO-Plus: we select it via PYTHONPATH (the eval command in the VLANeXt README does the same).
#    That way, which 'libero' gets imported depends on a single environment variable.
%cd /content
!git clone -q https://github.com/DravenALG/VLANeXt && echo "VLANeXt ok"
%cd /content/VLANeXt
!mkdir -p third_party && git clone -q https://github.com/sylvestf/LIBERO-plus.git third_party/LIBERO-plus && echo "LIBERO-plus ok"
!git -C /content/VLANeXt log -1 --format='VLANeXt %h %ci' && git -C third_party/LIBERO-plus log -1 --format='LIBERO-plus %h %ci'
!pip install -q "transformers==5.5.0" "diffusers==0.36.0" accelerate huggingface_hub sentencepiece h5py opencv-python "imageio[ffmpeg]" \
    "robosuite==1.4.1" "mujoco==3.8.1" "bddl==1.0.1" future easydict cloudpickle gym pyyaml wandb tqdm tensorflow-datasets rlds \
    wand scikit-image scipy 2>&1 | tail -2
# usd-core from extra_requirements.txt is not in the eval import chain (we searched for a pxr import and found none) → we do not install it.
!pip list 2>/dev/null | grep -i -E "^libero|hf-libero" || echo "no libero package in pip (correct)"

In [ ]:
# 4) LIBERO-Plus config: a SEPARATE folder (~/.libero_plus) with the paths pointing explicitly at the repo copy.
#    If there is no config, LIBERO asks via input() on first import → the notebook freezes; so we write the file beforehand.
import os, yaml, pathlib
PLUS = "/content/VLANeXt/third_party/LIBERO-plus"
lib = f"{PLUS}/libero/libero"
cfg_dir = pathlib.Path("~/.libero_plus").expanduser(); cfg_dir.mkdir(exist_ok=True)
yaml.safe_dump({"benchmark_root": lib, "bddl_files": f"{lib}/bddl_files", "init_states": f"{lib}/init_files",
                "datasets": f"{PLUS}/libero/datasets", "assets": f"{lib}/assets"}, open(cfg_dir / "config.yaml", "w"))
os.environ["LIBERO_CONFIG_PATH"] = str(cfg_dir)      # COMMON_ISSUES #6: must NOT be an empty string
os.environ["LIBERO_PLUS_ROOT"] = PLUS                # Lecture 16 patch 1
os.environ["PYTHONPATH"] = PLUS
print(open(cfg_dir / "config.yaml").read())

In [ ]:
# 5) Patches: Lecture 12 (sdpa, mmap, VLANEXT_TASK_IDS) + Lecture 16 (json path, VLANEXT_INSTR_OVERRIDE, VLANEXT_RUN_TAG, log, np.float_)
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/vlanext_t4_patch.py && curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/vlanext_plus_patch.py
!curl -s -o config/t4_plus_config.yaml https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/t4_plus_config.yaml
!python vlanext_t4_patch.py /content/VLANeXt && python vlanext_plus_patch.py /content/VLANeXt

In [ ]:
# 6) Assets: 6.4 GB zip (HF dataset Sylvest/LIBERO-plus) → libero/libero/assets/. Delete the zip after unpacking.
import time; t0 = time.time()
!hf download --repo-type dataset Sylvest/LIBERO-plus assets.zip --local-dir {lib}
!cd {lib} && unzip -q -o assets.zip && rm assets.zip
!ls {lib}/assets && du -sh {lib}/assets
print("time %.1f min" % ((time.time() - t0) / 60))
for d in ["articulated_objects", "new_objects", "scenes", "stable_hope_objects", "stable_scanned_objects", "textures", "turbosquid_objects"]:
    print(("OK   " if os.path.isdir(f"{lib}/assets/{d}") else "MISSING"), d)

In [ ]:
# 7) Checkpoint — the same file as in Lecture 12 (trained on LIBERO, will be tested on LIBERO-Plus with perturbations it has never seen)
!hf download DravenALG/VLANeXt VLANeXt_libero_spatial.pt --local-dir checkpoints && ls -lh checkpoints

In [ ]:
# 8) Smoke test (NO model): is the right 'libero' imported, are there 2402 tasks, what is task 988's command, does the env open?
!cd /tmp && python -c "import libero, libero.libero as L; print('libero path:', list(libero.__path__)); \
from libero.libero import benchmark; s = benchmark.get_benchmark_dict()['libero_spatial'](); print('n_tasks =', s.n_tasks); \
t = s.get_task(988); print('988:', t.name); print('language:', repr(t.language)); print('init:', s.get_task_init_states(988).shape)"
!cd /tmp && python -c "from libero.libero import benchmark, get_libero_path; from libero.libero.envs import OffScreenRenderEnv; import os; \
s = benchmark.get_benchmark_dict()['libero_spatial'](); t = s.get_task(988); \
env = OffScreenRenderEnv(bddl_file_name=os.path.join(get_libero_path('bddl_files'), t.problem_folder, t.bddl_file), camera_heights=128, camera_widths=128); \
env.seed(0); env.reset(); o = env.set_init_state(s.get_task_init_states(988)[0]); print('env ok, agentview', o['agentview_image'].shape); env.close()"

In [ ]:
# 9) Task selection: the 3 easiest of the 46 Language variants of base task 0 (between the plate and the ramekin)
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/select_plus_tasks.py
!python select_plus_tasks.py {PLUS} --suite libero_spatial --base 0 --category Language --pick 3 --json selection.json
import json; sel = json.load(open("selection.json")); IDS = sel["task_ids"]; CLEAN = sel["clean_language"]; print(IDS, "|", CLEAN)

In [ ]:
# 10) Conditions. Each condition = a separate eval that reloads the model (~3 min loading + 3 episodes).
#     rewritten : LIBERO-Plus's own (LLM-rewritten) command                      → NO override
#     empty     : empty command                                                  → VLANEXT_INSTR_OVERRIDE=""
#     clean     : (optional) the original LIBERO command, same scene + same init → override = CLEAN
#     tr        : (optional) the sp0_tr sentence from the Lecture 15 dataset
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/parse_plus_log.py
import subprocess, glob, os
TR = "tabak ile ramekinin arasındaki siyah kaseyi al ve tabağın üstüne koy"   # datasets/01-libero-spatial-tr-mini, sp0_tr
def run(cond, override=None, lang="en"):
    env = dict(os.environ, VLANEXT_TASK_IDS=IDS, VLANEXT_RUN_TAG="_" + cond, CUDA_VISIBLE_DEVICES="0", MUJOCO_EGL_DEVICE_ID="0")
    env.pop("VLANEXT_INSTR_OVERRIDE", None)
    if override is not None: env["VLANEXT_INSTR_OVERRIDE"] = override
    with open(f"stdout_{cond}.txt", "w") as f:
        subprocess.run(["python", "-m", "scripts.libero_plus_bench_eval", "--config", "config/t4_plus_config.yaml"],
                       env=env, stdout=f, stderr=subprocess.STDOUT)
    logs = sorted(glob.glob(f"checkpoints/libero_spatial_checkpoint_*_libero_plus_{cond}*/eval.log"))
    print(cond, "→", logs[-1] if logs else "NO LOG, look at the end of the file stdout_" + cond + ".txt")
    if logs:
        r = subprocess.run(["python", "parse_plus_log.py", logs[-1], "--run-id", "vlanext_plus_t4_" + cond,
                            "--condition", cond, "--lang", lang, "--csv", "results/runs.csv"], capture_output=True, text=True)
        print(r.stdout, r.stderr)

run("rewritten")
run("empty", override="", lang="none")

In [ ]:
# 10b) Optional (if you have time): clean EN and the Turkish command on the same 3 scenes
run("clean", override=CLEAN)
run("tr", override=TR, lang="tr")

In [ ]:
# 11) Summary + videos. Rates from 3 episodes are not a 'result' (Lecture 12: the Wilson 95% interval of 3/3 is 43.8–100) — they are proof that the pipeline works.
import pandas as pd
df = pd.read_csv("results/runs.csv"); print(df.groupby("condition")[["n_episodes"]].sum().join(df.groupby("condition")["success_rate"].mean()))
from IPython.display import Video, display
vids = sorted(glob.glob("checkpoints/*_libero_plus_*/*.mp4")); print(*vids, sep="\n")
display(Video(vids[0], embed=True, width=384))

## Next
- Download `results/runs.csv`, the four `eval.log` files and the video of one failed episode → `projects/04-libero-plus-kurulum/` (create it from the template).
- Write in the README: the unpacked size of the assets (cell 6), the duration of each cell, the `libero path` line from the smoke test, the episode results of `rewritten` / `empty` (and `clean` / `tr` if you ran them).
- In Lecture 17 we will separate the types of these 46 rewrites (R1 distraction, R2 commonsense description, R3 reasoning chain) in code and in data.